In [1]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": -6,
	"longitude": 35,
	"hourly": ["temperature_2m", "relative_humidity_2m", "rain", "soil_temperature_0cm", "soil_temperature_6cm", "soil_temperature_18cm"],
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_rain = hourly.Variables(2).ValuesAsNumpy()
hourly_soil_temperature_0cm = hourly.Variables(3).ValuesAsNumpy()
hourly_soil_temperature_6cm = hourly.Variables(4).ValuesAsNumpy()
hourly_soil_temperature_18cm = hourly.Variables(5).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["rain"] = hourly_rain
hourly_data["soil_temperature_0cm"] = hourly_soil_temperature_0cm
hourly_data["soil_temperature_6cm"] = hourly_soil_temperature_6cm
hourly_data["soil_temperature_18cm"] = hourly_soil_temperature_18cm

hourly_dataframe = pd.DataFrame(data = hourly_data)
hourly_dataframe


Coordinates: -6.0°N 35.0°E
Elevation: 1035.0 m asl
Timezone difference to GMT+0: 0s


,date,temperature_2m,relative_humidity_2m,rain,soil_temperature_0cm,soil_temperature_6cm,soil_temperature_18cm
0,2026-03-17 00:00:00+00:00,20.183001,86.0,0.0,18.782999,20.782999,22.833000
1,2026-03-17 01:00:00+00:00,19.782999,89.0,0.0,18.583000,20.532999,22.683001
2,2026-03-17 02:00:00+00:00,19.483000,91.0,0.0,18.483000,20.333000,22.483000
3,2026-03-17 03:00:00+00:00,19.083000,94.0,0.0,18.483000,20.183001,22.333000
4,2026-03-17 04:00:00+00:00,19.083000,95.0,0.0,18.833000,20.132999,22.132999
...,...,...,...,...,...,...,...
163,2026-03-23 19:00:00+00:00,20.833000,90.0,0.0,20.233000,22.333000,23.632999
164,2026-03-23 20:00:00+00:00,20.483000,92.0,0.0,19.933001,21.933001,23.483000
165,2026-03-23 21:00:00+00:00,20.282999,93.0,0.0,19.733000,21.632999,23.333000
166,2026-03-23 22:00:00+00:00,20.132999,93.0,0.0,19.483000,21.382999,23.183001
